# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [21]:
import pandas as pd

---
## 2. Load data

In [22]:
df = pd.read_csv("/content/clean_data_after_eda.csv")
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [23]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [24]:
price_df = pd.read_csv('/content/price_data (1).csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [25]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

In [26]:
price_features = price_df.groupby("id").agg(
    avg_off_peack_var=("price_off_peak_var", "mean"),
    avg_peak_var=("price_peak_var", "mean"),
    avg_mid_peak_fix=("price_mid_peak_fix", "mean"),
    std_off_peak_fix=("price_off_peak_fix", "std"), # Corrected column and function
    std_peak_var=("price_peak_var", "std"),       # Corrected column and function
    std_mid_peak_fix=("price_mid_peak_fix", "std"), # Corrected column and function
).reset_index()
price_features.head()

,id,avg_off_peack_var,avg_peak_var,avg_mid_peak_fix,std_off_peak_fix,std_peak_var,std_mid_peak_fix
0,0002203ffbb812588b632b9e628cc38d,0.124338,0.103794,16.280694,6.341481e-02,0.001989,0.025366
1,0004351ebdd665e6ee664792efc4fd13,0.146426,0.000000,0.000000,8.753223e-02,0.000000,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.181558,0.000000,0.000000,7.723930e-01,0.000000,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,0.118757,0.098292,16.258971,8.507958e-02,0.002580,0.034032
4,00114d74e963e47177db89bc70108537,0.147926,0.000000,0.000000,5.908392e-07,0.000000,0.000000


In [27]:
diff = diff.merge(price_features,on='id', how= 'left')
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power,avg_off_peack_var,avg_peak_var,avg_mid_peak_fix,std_off_peak_fix,std_peak_var,std_mid_peak_fix
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916,0.124338,0.103794,16.280694,6.341481e-02,0.001989,0.025366
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779,0.146426,0.000000,0.000000,8.753223e-02,0.000000,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000,0.181558,0.000000,0.000000,7.723930e-01,0.000000,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916,0.118757,0.098292,16.258971,8.507958e-02,0.002580,0.034032
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001,0.147926,0.000000,0.000000,5.908392e-07,0.000000,0.000000


In [28]:
diff['price-range'] = (
    diff['avg_peak_var'] -
    diff['avg_off_peack_var']
)

diff[['id', 'avg_off_peack_var',
      'avg_peak_var',
      'price-range']].head()

,id,avg_off_peack_var,avg_peak_var,price-range
0,0002203ffbb812588b632b9e628cc38d,0.124338,0.103794,-0.020545
1,0004351ebdd665e6ee664792efc4fd13,0.146426,0.000000,-0.146426
2,0010bcc39e42b3c2131ed2ce55246e3c,0.181558,0.000000,-0.181558
3,0010ee3855fdea87602a5b7aba8e42de,0.118757,0.098292,-0.020465
4,00114d74e963e47177db89bc70108537,0.147926,0.000000,-0.147926


In [29]:
diff[['offpeak_diff_dec_january_energy', 'offpeak_diff_dec_january_power']].head()

,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,-0.006192,0.162916
1,-0.004104,0.177779
2,0.050443,1.500000
3,-0.010018,0.162916
4,-0.003994,-0.000001


In [30]:
price_df['peak_offpeak_diff'] = (
    price_df['price_peak_var'] -
    price_df['price_off_peak_var']
)

price_df[['price_peak_var',
          'price_off_peak_var',
          'price_off_peak_var']].head()

,price_peak_var,price_off_peak_var,price_off_peak_var
0,0.0,0.151367,0.151367
1,0.0,0.151367,0.151367
2,0.0,0.151367,0.151367
3,0.0,0.149626,0.149626
4,0.0,0.149626,0.149626


In [31]:
df['consumption_diffrence'] = (
    df['cons_12m'] -
    (df['cons_last_month'] * 12)
)

df[['cons_12m',
'cons_last_month',
'consumption_diffrence']] . head()

,cons_12m,cons_last_month,consumption_diffrence
0,0,0,0
1,4660,0,4660
2,544,0,544
3,1584,0,1584
4,4425,526,-1887


In [32]:
df['consumption_ratio'] = (
    df['cons_last_month'] /
    (df['cons_12m'] + 1e-6)
)

df[['cons_12m',
'cons_last_month',
'consumption_ratio']] .head()

,cons_12m,cons_last_month,consumption_ratio
0,0,0,0.00000
1,4660,0,0.00000
2,544,0,0.00000
3,1584,0,0.00000
4,4425,526,0.11887


**FEATURE ENGINERING MODELING**

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as datetime

2. **LOAD DATA**

In [38]:
df = pd.read_csv('/content/data_for_predictions.csv')
df.drop(columns=["Unnamed: 0"], inplace=True)
df.head()

,id,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,...,months_modif_prod,months_renewal,channel_MISSING,channel_ewpakwlliwisiwduibdlfmalxowmwpci,channel_foosdfpfkusacimwkcsosbicdxkicaua,channel_lmkebamcaaclubfxadlmueccxoimlema,channel_usilxuppasemubllopkaafesmlibmsdf,origin_up_kamkkxfxxuwbdslkwifmmcsiusiuosws,origin_up_ldkssxwpmemidmecebumciepifcamkci,origin_up_lxidpiddsbxsbosboudacockeimpuepw
0,24011ae4ebbe3035111d65fa7c15bc57,0.000000,4.739944,0.000000,0.000000,0.0,0.444045,0.114481,0.098142,40.606701,...,2,6,0,0,1,0,0,0,0,1
1,d29c2c54acc38ff3c0614d0a653813dd,3.668479,0.000000,0.000000,2.280920,0.0,1.237292,0.145711,0.000000,44.311378,...,76,4,1,0,0,0,0,1,0,0
2,764c75f661154dac3a6c254cd082ea7d,2.736397,0.000000,0.000000,1.689841,0.0,1.599009,0.165794,0.087899,44.311378,...,68,8,0,0,1,0,0,1,0,0
3,bba03439a292a1e166f80264c16191cb,3.200029,0.000000,0.000000,2.382089,0.0,1.318689,0.146694,0.000000,44.311378,...,69,9,0,0,0,1,0,1,0,0
4,149d57cf92fc41cf94415803a877cb4b,3.646011,0.000000,2.721811,2.650065,0.0,2.122969,0.116900,0.100015,40.606701,...,71,9,1,0,0,0,0,1,0,0


# 3. Modeling
This dataset contains that have been engineered and are ready to be trained  

In [39]:
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Data Sampling
The data is split into training and test samples. The reason for this is so that we can stimulate real life situations  y generation stimulations for our test samples, withput showing pretedictive models the data these data points. This allows us to see how well our model is able to generaklise new data, which is critical

A typical % for tarining is between 20-30 but in this case we are going to use 75-25% split bewteen tarin and test rescpectfully


In [41]:
# make a copy for our data

train_df = df.copy()

#seperate traget variable frem independent varible
y = df["churn"]
x = df.drop(columns=['id','churn'])
print(x.shape)
print(y.shape)

(14606, 61)
(14606,)


In [43]:
x_train, x_test, y_train, y_test, = train_test_split(x,y, test_size=0.25, random_state=42)

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(10954, 61)
(10954,)
(3652, 61)
(3652,)


### Model training

Once again, we are using a `Random Forest` classifier in this example. A Random Forest sits within the category of `ensemble` algorithms because internally the `Forest` refers to a collection of `Decision Trees` which are tree-based learning algorithms. As the data scientist, you can control how large the forest is (that is, how many decision trees you want to include).

The reason why an `ensemble` algorithm is powerful is because of the laws of averaging, weak learners and the central limit theorem. If we take a single decision tree and give it a sample of data and some parameters, it will learn patterns from the data. It may be overfit or it may be underfit, but that is now our only hope, that single algorithm.

With `ensemble` methods, instead of banking on 1 single trained model, we can train 1000's of decision trees, all using different splits of the data and learning different patterns. It would be like asking 1000 people to all learn how to code. You would end up with 1000 people with different answers, methods and styles! The weak learner notion applies here too, it has been found that if you train your learners not to overfit, but to learn weak patterns within the data and you have a lot of these weak learners, together they come together to form a highly predictive pool of knowledge! This is a real life application of many brains are better than 1.

Now instead of relying on 1 single decision tree for prediction, the random forest puts it to the overall views of the entire collection of decision trees. Some ensemble algorithms using a voting approach to decide which prediction is best, others using averaging.

As we increase the number of learners, the idea is that the random forest's performance should converge to its best possible solution.

Some additional advantages of the random forest classifier include:

- The random forest uses a rule-based approach instead of a distance calculation and so features do not need to be scaled
- It is able to handle non-linear parameters better than linear based models

On the flip side, some disadvantages of the random forest classifier include:

- The computational power needed to train a random forest on a large dataset is high, since we need to build a whole ensemble of estimators.
- Training time can be longer due to the increased complexity and size of thee ensemble

In [46]:
# model training
model = RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(x_train, y_train)

RandomForestClassifier(random_state=42)

In [47]:
y_pred = model.predict(x_test)
print(y_pred[:10])

[0 0 0 0 0 0 0 0 0 0]


# Evaluation


In [49]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

precision = precision_score(y_test, y_pred, zero_division=0)
print("Precision:", precision)

recall = recall_score(y_test, y_pred, zero_division=0)
print("Recall:", recall)

f1_scr = f1_score(y_test, y_pred, zero_division=0)
print("f1 score:", f1_scr)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test,y_pred))

Accuracy: 0.9030668127053669
Precision: 0.7142857142857143
Recall: 0.0546448087431694
f1 score: 0.10152284263959391

Confusion Matrix:
[[3278    8]
 [ 346   20]]


# Model Evaluation

i evaluted the Random Forest model usiong accurarcy, precission, recall andf` score. These metrics where choosen because accuracy alone may not provide a complete picture of the models performance, expecially when the class are imbalanced. Precision measures how many of the cases are predicted as positive were actually positive, while recall measures how manyof the actual positive cases were correctly identified. The F1 score provides a balance between precision and recall.

The model achived an accuracy of approximately 90.31% and a precision of 71.43%. however the recall was only 5.46% and f1 score was 10.15%. This indicates that although the modelcorrectly classifed a large proportion of the overall observations, it struggled to idntofed the positive class.

Based on these result, i would not consider the models performance fully satisfactory. tHe congusion matrics shows 364 false engetaive compared withonly 20 true positives, meaning that mamy true positive caes where missed. futher improvement such a adressing class imbalnce and turnig the models ability to identfy positve cases.